# Building block — scale testing — configured by one file

> **Building block.** A minimal, copy-pasteable block (config explanation +
> run + save) and nothing else. Paste it together with other blocks — e.g.
> `building_block_consent.ipynb` to get a `participant_id` — to assemble your
> own experiment notebook.

This notebook runs a **scale test** and saves the ratings. On each screen the
participant hears **one stimulus** (a single play button) and rates it on
every scale stacked below — each scale is either a row of **Likert-style
buttons** or a **slider**, as configured. This is the classic setup for
semantic-differential / attribute-rating experiments.

Like the other whispy experiments it is **config-driven**: the whole thing is
described in [`configs/scale_testing.yml`](../configs/scale_testing.yml); only
the shared look comes from [`configs/design.yml`](../configs/design.yml). You
should not need to edit any Python below.

That one file is read by three consumers:

| block in the YAML | used by | configures |
|---|---|---|
| `SoundDevice:` | the audio handler | which WAV plays for each stimulus id |
| `ui:` / `questions:` | the ScaleTest window | window size, wording, and the rating scales |
| `trial:` / `experiment:` | this notebook | the prompt and which stimuli are rated (blocks/sections) |

If you haven't installed the project yet, run this from the repo root:
`pip install -e .`

In [1]:
import whispy
import pandas as pd
from pathlib import Path
from whispy.interfaces import SoundDevice
from whispy.ui import ScaleTest
from whispy.utils import read_config

config_path = Path('..') / 'configs' / 'scale_testing.yml'   # <- the one file you edit
stimuli_dir = Path('demo_stimuli/scale_testing')             # <- folder with your WAVs

cfg = read_config(config_path)
handler = SoundDevice(config_path, stimuli_dir, loop=False)   # reads SoundDevice:
print('available stimulus ids:', list(handler.stimuli.keys()))

available stimulus ids: ['smooth_dark', 'smooth_bright', 'rough_dark', 'rough_bright']


## See your scale test at a glance

Open [`configs/scale_testing.yml`](../configs/scale_testing.yml) to edit every
setting (it has inline comments). The cell below prints the current values and
the questions, so you can review the design without leaving the notebook.

In [2]:
print('ui        :', cfg['ui'])
print('trial     :', cfg['trial'])
print('experiment:', cfg['experiment'])
print()
print('questions (each is one scale on every screen):')
for i, q in enumerate(cfg['questions'], start=1):
    rng = q.get('scale_range', [1, 5])
    print(f"  {i}. {q.get('id', f'q{i}')}: {q.get('prompt')}"
          f"  [{q.get('interaction_method', 'buttons')}, {rng[0]}..{rng[1]}]")

ui        : {'content_area_size': [70, 90], 'button_fontsize': 8, 'submit_button_fontsize': 14, 'play_button_text': '► Play', 'submit_hint': 'Press Space (or click Play) to listen, rate every scale, then press Enter to submit.', 'submit_button_text': 'Submit ratings', 'show_progress': True, 'progress_text': 'Trial {current} of {total}'}
trial     : {'task': 'Listen to the sound and rate it on every scale below.'}
experiment: [{'block': [{'block_name': 'timbre'}, {'section': {'section_name': 'synthetic tones', 'test': ['smooth_dark', 'smooth_bright', 'rough_dark', 'rough_bright']}}]}]

questions (each is one scale on every screen):
  1. roughness: How **rough** does the sound feel?  [buttons, 1..5]
  2. brightness: How **bright** is the sound?  [slider, 0..100]
  3. pleasantness: How **pleasant** is the sound?  [buttons, 1..5]
  4. sharpness: Compared to a neutral tone, is the sound **softer or sharper**?  [buttons, -5..5]


## Using your own stimuli

The WAVs in `examples/demo_stimuli/scale_testing/` are **only a demo** (run
`python examples/demo_stimuli/scale_testing/generate_scale_testing_stimuli.py`
to regenerate them). For a real experiment you swap in your own files — you
only touch the config and `stimuli_dir`, never the Python.

1. **Point `stimuli_dir`** (in the setup cell) at the folder holding your WAVs.
2. **Map ids to files** under `SoundDevice:` in
   [`configs/scale_testing.yml`](../configs/scale_testing.yml), then list them
   under `experiment:` in the section's `test:` list:
   ```yaml
   SoundDevice:
     violin_dry: { file: violin_dry.wav }
     violin_hall: { file: violin_hall.wav }
   experiment:
     - block:
         - block_name: my block
         - section:
             section_name: violins
             test: [violin_dry, violin_hall]
   ```
3. **Design the scales** under `questions:` — per question choose
   `interaction_method` (`buttons` or `slider`), the `scale_range`, and
   optional `labels` (two entries = endpoints; for buttons, one per step =
   a label under each button).

**Safety constraints** (checked when the handler loads in the setup cell — a
violation raises a `ValueError`):

- **No clipping** — every sample must satisfy `|amplitude| < 1` (normalise with headroom, e.g. ~0.7).
- **One sampling rate** — every file must share the same sampling rate.
- **All ids must exist** — every id used under `experiment:` must be defined under `SoundDevice:`.

## Build the screens

`whispy.ExperimentScheduler` randomizes the `experiment:` block
(blocks/sections/stimuli, exactly like MUSHRA). Scale testing then presents
**one stimulus per screen**, so each scheduler screen (which carries a
section's stimuli as a list) is expanded into consecutive single-stimulus
screens, and the progress counter is recomputed over the expanded list.

In [3]:
scheduler = whispy.ExperimentScheduler(experiment=cfg['experiment'])

screens = []
for section_screen in scheduler:
    for stimulus in list(section_screen['test']):
        screens.append({
            'stimulus': stimulus,
            'task': cfg['trial']['task'],
            'block': section_screen['block'],
            'section': section_screen['section'],
            'block_name': section_screen['block_name'],
            'section_name': section_screen['section_name'],
        })
for i, screen in enumerate(screens, start=1):
    screen['trial_id'] = i
    # drives the "Trial X of Y" progress bar (ui.show_progress)
    screen['progress'] = {'current': i, 'total': len(screens)}

print(f'{len(screens)} screens (one stimulus each):')
for screen in screens:
    print(f"  {screen['trial_id']:>2}. {screen['stimulus']}")

4 screens (one stimulus each):
   1. smooth_dark
   2. rough_dark
   3. rough_bright
   4. smooth_bright


## Run it

The loop below presents every screen, reusing **one window** for the whole
experiment (the first screen opens it; later screens swap their content into
the same window via `parent=host`, so it stays fullscreen with no reload). The
window is closed in a `finally` block when the run finishes.

On each screen: press **Space** (or click **Play**) to hear the stimulus,
answer every scale (click a button, or drag/click a slider), then press
**Enter** to submit. Submitting requires the stimulus to have been played and
every scale to be answered.

In [ ]:
host = None
results = None

try:
    for screen in screens:
        scale_test = ScaleTest(screen=screen, stimuli_handler=handler,
                               scale_test_config=config_path,
                               blocking=True, parent=host)
        if host is None:
            host = scale_test    # first screen owns the shared window
        results = scale_test.get_results(results)
finally:
    if host is not None:
        host.close()

results

## Results

`results` is **long-form**: one row per question per stimulus (`stimulus`,
`question`, `answer`, `scale_min`, `scale_max`, `rt`, ...). The cell below
pivots it into a stimulus × question table of the given answers.

In [ ]:
print(results.pivot_table(index='stimulus', columns='question',
                          values='answer', aggfunc='mean').round(1).to_string())

## Save the results

Save the results to a CSV in a `results/` folder (created next to this
notebook). The file name always carries a **timestamp**. If a
`participant_id` is in scope (set by a consent block earlier in the notebook)
it is included (`<name>_<id>_<timestamp>.csv`); otherwise an iterating
fallback number is used (`<name>_<NNN>_<timestamp>.csv`). Existing files are
never overwritten.

In [ ]:
from whispy.utils import save_results

# `participant_id` is picked up automatically if a consent block earlier in
# this notebook set it; otherwise it is None and the file name uses an
# iterating fallback number.
participant_id = globals().get('participant_id')
results_path = save_results(results, 'scale_testing', participant_id=participant_id)
print('saved results to', results_path.resolve())